The basic question becomes:
Are certain books popular across users from several countries/regions, rather than primarily within one geographic market?

In [0]:
SELECT
    Location,
    COUNT(*) AS user_count
FROM users
GROUP BY Location
ORDER BY user_count DESC
LIMIT 50;

Location,user_count
"london, england, united kingdom",2506
"toronto, ontario, canada",2250
"sydney, new south wales, australia",1744
"melbourne, victoria, australia",1708
"portland, oregon, usa",1629
"chicago, illinois, usa",1526
"seattle, washington, usa",1484
"new york, new york, usa",1411
"madrid, madrid, spain",1400
"vancouver, british columbia, canada",1359


In [0]:
SELECT
    COUNT(*) AS total_users,
    COUNT(DISTINCT Location) AS distinct_locations,
    SUM(CASE
        WHEN Location IS NULL OR TRIM(Location) = ''
        THEN 1 ELSE 0
    END) AS missing_locations
FROM users;

total_users,distinct_locations,missing_locations
278859,57329,0


In [0]:
SELECT
    LOWER(TRIM(SPLIT(Location, ',')[2])) AS country,
    COUNT(*) AS user_count
FROM users
WHERE SIZE(SPLIT(Location, ',')) >= 3
GROUP BY LOWER(TRIM(SPLIT(Location, ',')[2]))
ORDER BY user_count DESC
LIMIT 100;

country,user_count
usa,139188
canada,21558
united kingdom,18286
germany,17022
spain,13096
australia,11724
italy,11241
,4610
france,3442
portugal,3306


In [0]:
WITH extracted_countries AS (
    SELECT
        LOWER(TRIM(SPLIT(Location, ',')[2])) AS country
    FROM users
    WHERE SIZE(SPLIT(Location, ',')) >= 3
)

SELECT
    COUNT(DISTINCT country) AS distinct_countries,
    COUNT(*) AS users_with_3_part_location,
    SUM(
        CASE
            WHEN country IS NULL
              OR country = ''
              OR country = 'n/a'
            THEN 1
            ELSE 0
        END
    ) AS invalid_country_users
FROM extracted_countries;

distinct_countries,users_with_3_part_location,invalid_country_users
1016,278193,4700


In [0]:
WITH extracted_countries AS (
    SELECT
        LOWER(TRIM(SPLIT(Location, ',')[2])) AS country
    FROM users
    WHERE SIZE(SPLIT(Location, ',')) >= 3
)

SELECT
    country,
    COUNT(*) AS user_count
FROM extracted_countries
WHERE country IS NOT NULL
  AND country <> ''
  AND country <> 'n/a'
GROUP BY country
ORDER BY user_count ASC, country
LIMIT 100;

country,user_count
中国,1
美国,1
*,1
-,1
-------,1
01776,1
02458,1
13 - 10ºesq.,1
19104,1
23232,1


these results confirm that the tail of the location data is very dirty, which is useful to discover before we build anything on it.

278,193 users whose location can be split into at least 3 components.

- 1,016 distinct “country” strings — obviously far more than the number of actual countries.
- 4,700 users where the extracted country is blank/n/a.
- The rare-value output contains things like *, -, 01776, 02458, 19104, 23232, 13 - 10ºesq., and encoded values such as &#20013;&#22269;.

Our actual goal is international relevance, not perfect location cleaning. So instead of trying to repair every obscure value, we can retain countries we can confidently recognize and classify the rest as unknown.

I think our international-relevance metric should eventually reward geographic diversity.
For example:
Book A
North America   90%
Europe           8%
Other            2%

Book B
North America   45%
Europe          30%
Oceania         10%
South America    8%
Asia             7%

Even if Book A has more total ratings, Book B gives stronger evidence of cross-regional appeal.

Next step: create a country → continent mapping


In [0]:
WITH extracted_countries AS (
    SELECT
        LOWER(TRIM(SPLIT(Location, ',')[2])) AS country
    FROM users
    WHERE SIZE(SPLIT(Location, ',')) >= 3
),

country_counts AS (
    SELECT
        country,
        COUNT(*) AS user_count
    FROM extracted_countries
    WHERE country IS NOT NULL
      AND country <> ''
      AND country <> 'n/a'
    GROUP BY country
)

SELECT
    COUNT(*) AS countries_with_100_plus_users,
    SUM(user_count) AS users_covered,
    ROUND(
        100.0 * SUM(user_count) / 278859,
        2
    ) AS percentage_of_all_users
FROM country_counts
WHERE user_count >= 100;

countries_with_100_plus_users,users_covered,percentage_of_all_users
57,269134,96.51


“For geographic analysis, I focused on sufficiently represented locations and mapped recognized country values to continents. Rare or malformed locations were classified as unknown rather than making potentially incorrect geographic assumptions.”

“The raw dataset contained 1,016 distinct country strings, including malformed and inconsistent values. However, the 57 country values represented by at least 100 users cover 96.51% of the entire user base. Therefore, I focused the geographic analysis on these sufficiently represented locations rather than attempting unreliable corrections of rare malformed values.”

In [0]:
WITH extracted_countries AS (
    SELECT
        LOWER(TRIM(SPLIT(Location, ',')[2])) AS country
    FROM users
    WHERE SIZE(SPLIT(Location, ',')) >= 3
),

country_counts AS (
    SELECT
        country,
        COUNT(*) AS user_count
    FROM extracted_countries
    WHERE country IS NOT NULL
      AND country <> ''
      AND country <> 'n/a'
    GROUP BY country
)

SELECT
    country,
    user_count
FROM country_counts
WHERE user_count >= 100
ORDER BY user_count DESC;

country,user_count
usa,139188
canada,21558
united kingdom,18286
germany,17022
spain,13096
australia,11724
italy,11241
france,3442
portugal,3306
new zealand,3071


yugoslavia is a historical country, but for this analysis we can map it to Europe rather than trying to infer a modern successor state

In [0]:
CREATE OR REPLACE TEMP VIEW country_continent_mapping AS

SELECT * FROM VALUES
    ('usa', 'United States', 'North America'),
    ('canada', 'Canada', 'North America'),
    ('united kingdom', 'United Kingdom', 'Europe'),
    ('england', 'United Kingdom', 'Europe'),
    ('germany', 'Germany', 'Europe'),
    ('spain', 'Spain', 'Europe'),
    ('australia', 'Australia', 'Oceania'),
    ('italy', 'Italy', 'Europe'),
    ('france', 'France', 'Europe'),
    ('portugal', 'Portugal', 'Europe'),
    ('new zealand', 'New Zealand', 'Oceania'),
    ('netherlands', 'Netherlands', 'Europe'),
    ('switzerland', 'Switzerland', 'Europe'),
    ('brazil', 'Brazil', 'South America'),
    ('china', 'China', 'Asia'),
    ('sweden', 'Sweden', 'Europe'),
    ('india', 'India', 'Asia'),
    ('austria', 'Austria', 'Europe'),
    ('malaysia', 'Malaysia', 'Asia'),
    ('argentina', 'Argentina', 'South America'),
    ('singapore', 'Singapore', 'Asia'),
    ('finland', 'Finland', 'Europe'),
    ('belgium', 'Belgium', 'Europe'),
    ('mexico', 'Mexico', 'North America'),
    ('denmark', 'Denmark', 'Europe'),
    ('ireland', 'Ireland', 'Europe'),
    ('philippines', 'Philippines', 'Asia'),
    ('turkey', 'Turkey', 'Asia'),
    ('poland', 'Poland', 'Europe'),
    ('pakistan', 'Pakistan', 'Asia'),
    ('greece', 'Greece', 'Europe'),
    ('iran', 'Iran', 'Asia'),
    ('romania', 'Romania', 'Europe'),
    ('chile', 'Chile', 'South America'),
    ('israel', 'Israel', 'Asia'),
    ('south africa', 'South Africa', 'Africa'),
    ('indonesia', 'Indonesia', 'Asia'),
    ('norway', 'Norway', 'Europe'),
    ('japan', 'Japan', 'Asia'),
    ('croatia', 'Croatia', 'Europe'),
    ('south korea', 'South Korea', 'Asia'),
    ('nigeria', 'Nigeria', 'Africa'),
    ('slovakia', 'Slovakia', 'Europe'),
    ('czech republic', 'Czech Republic', 'Europe'),
    ('russia', 'Russia', 'Europe'),
    ('yugoslavia', 'Yugoslavia', 'Europe'),
    ('hong kong', 'Hong Kong', 'Asia'),
    ('taiwan', 'Taiwan', 'Asia'),
    ('costa rica', 'Costa Rica', 'North America'),
    ('slovenia', 'Slovenia', 'Europe'),
    ('peru', 'Peru', 'South America'),
    ('vietnam', 'Vietnam', 'Asia'),
    ('venezuela', 'Venezuela', 'South America'),
    ('egypt', 'Egypt', 'Africa'),
    ('bulgaria', 'Bulgaria', 'Europe'),
    ('colombia', 'Colombia', 'South America'),
    ('luxembourg', 'Luxembourg', 'Europe')

AS mapping(country_raw, standardized_country, continent);

In [0]:
CREATE OR REPLACE TEMP VIEW user_geography AS

SELECT
    u.`User-ID`,
    LOWER(TRIM(try_element_at(SPLIT(u.Location, ','), 3))) AS country_raw,
    m.standardized_country,
    m.continent

FROM users u

LEFT JOIN country_continent_mapping m
    ON LOWER(TRIM(try_element_at(SPLIT(u.Location, ','), 3))) = m.country_raw;

In [0]:
SELECT
    continent,
    COUNT(*) AS user_count,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_mapped_users
FROM user_geography
WHERE continent IS NOT NULL
GROUP BY continent
ORDER BY user_count DESC;

continent,user_count,percentage_of_mapped_users
North America,161714,60.09
Europe,80058,29.75
Oceania,14795,5.50
Asia,8385,3.12
South America,3507,1.30
Africa,675,0.25


continue using explicit ratings only (Book-Rating > 0), because those represent actual expressed opinions and keep this analysis consistent with our quality analysis

In [0]:
CREATE OR REPLACE TEMP VIEW book_continent_stats AS
SELECT
    r.ISBN,
    g.continent,

    COUNT(*) AS explicit_ratings,

    COUNT(DISTINCT r.`User-ID`) AS unique_users,

    AVG(r.`Book-Rating`) AS avg_rating

FROM ratings r

INNER JOIN user_geography g
    ON r.`User-ID` = g.`User-ID`

WHERE r.`Book-Rating` > 0
  AND g.continent IS NOT NULL

GROUP BY
    r.ISBN,
    g.continent;

In [0]:
SELECT
    COUNT(DISTINCT s.continent) AS continents_reached,
    COUNT(*) AS number_of_books
FROM book_continent_stats s

INNER JOIN eligible_books e
    ON s.ISBN = e.ISBN

GROUP BY s.ISBN;

continents_reached,number_of_books
1,1
1,1
1,1
1,1
3,3
1,1
1,1
1,1
4,4
1,1


In [0]:
WITH book_reach AS (
    SELECT
        s.ISBN,
        COUNT(DISTINCT s.continent) AS continents_reached
    FROM book_continent_stats s

    INNER JOIN eligible_books e
        ON s.ISBN = e.ISBN

    GROUP BY s.ISBN
)

SELECT
    continents_reached,
    COUNT(*) AS number_of_books,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_books
FROM book_reach
GROUP BY continents_reached
ORDER BY continents_reached;

continents_reached,number_of_books,percentage_of_books
1,7795,53.63
2,4533,31.19
3,1701,11.70
4,465,3.20
5,40,0.28
6,1,0.01


In [0]:
CREATE OR REPLACE TEMP VIEW book_geographic_distribution AS
SELECT
    ISBN,
    continent,
    explicit_ratings,

    explicit_ratings * 1.0 /
        SUM(explicit_ratings) OVER (PARTITION BY ISBN)
        AS rating_share

FROM book_continent_stats;

In [0]:
CREATE OR REPLACE TEMP VIEW book_geographic_stats AS
SELECT
    ISBN,

    COUNT(*) AS continents_reached,

    SUM(explicit_ratings) AS mapped_explicit_ratings,

    MAX(rating_share) AS largest_continent_share,

    SUM(
        CASE
            WHEN rating_share >= 0.10 THEN 1
            ELSE 0
        END
    ) AS significant_continents

FROM book_geographic_distribution

GROUP BY ISBN;

continents_reached = continent has at least one rating.

significant_continents = continent contributes at least 10% of that book's mapped explicit ratings.

In [0]:
SELECT
    significant_continents,
    COUNT(*) AS number_of_books,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_books
FROM book_geographic_stats g
INNER JOIN eligible_books e
    ON g.ISBN = e.ISBN
GROUP BY significant_continents
ORDER BY significant_continents;

significant_continents,number_of_books,percentage_of_books
1,9894,68.07
2,3739,25.72
3,844,5.81
4,57,0.39
5,1,0.01


In [0]:
CREATE OR REPLACE VIEW workspace.default.book_geographic_reach AS
WITH book_location_reach AS (
    SELECT 
        r.ISBN,
        COUNT(DISTINCT u.Location) AS continents_reached
    FROM workspace.default.ratings r
    JOIN workspace.default.users u ON r.`User-ID` = u.`User-ID`
    WHERE r.ISBN IN (SELECT ISBN FROM workspace.default.book_quality_scores)
    GROUP BY r.ISBN
)
SELECT 
    continents_reached,
    COUNT(*) AS number_of_books,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage_of_books
FROM book_location_reach
GROUP BY continents_reached
ORDER BY continents_reached;

In [0]:
SELECT continent, COUNT(DISTINCT ISBN) as books, COUNT(*) as ratings
FROM workspace.default.book_continent_stats
GROUP BY continent;

continent,books,ratings
Other,340556,1149780


In [0]:
WITH book_reach AS (
    SELECT
        s.ISBN,
        COUNT(DISTINCT s.continent) AS continents_reached
    FROM workspace.default.book_continent_stats s
    INNER JOIN eligible_books e
        ON s.ISBN = e.ISBN
    GROUP BY s.ISBN
)

SELECT
    continents_reached,
    COUNT(*) AS number_of_books,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_books
FROM book_reach
GROUP BY continents_reached
ORDER BY continents_reached;

continents_reached,number_of_books,percentage_of_books
1,14535,100.00


In [0]:
SELECT
    bcs.ISBN,
    COUNT(DISTINCT bcs.continent) AS continents_reached,
    COLLECT_SET(bcs.continent) AS continents,
    r.total_ratings
FROM workspace.default.book_continent_stats bcs
INNER JOIN (
    SELECT ISBN, COUNT(*) AS total_ratings
    FROM workspace.default.ratings
    WHERE `Book-Rating` > 0
    GROUP BY ISBN
) r ON bcs.ISBN = r.ISBN
GROUP BY bcs.ISBN, r.total_ratings
ORDER BY continents_reached DESC, total_ratings DESC
LIMIT 30;

ISBN,continents_reached,continents,total_ratings
0316666343,1,List(Other),707
0971880107,1,List(Other),581
0385504209,1,List(Other),487
0312195516,1,List(Other),383
0679781587,1,List(Other),333
0060928336,1,List(Other),320
059035342X,1,List(Other),313
0142001740,1,List(Other),307
0446672211,1,List(Other),295
044023722X,1,List(Other),281


In [0]:
SELECT
    bcs.continent,
    COUNT(*) AS rows_count,
    COUNT(DISTINCT bcs.ISBN) AS distinct_isbns,
    COALESCE(SUM(r.total_ratings), 0) AS explicit_ratings
FROM workspace.default.book_continent_stats bcs
LEFT JOIN (
    SELECT ISBN, COUNT(*) AS total_ratings
    FROM workspace.default.ratings
    WHERE `Book-Rating` > 0
    GROUP BY ISBN
) r ON bcs.ISBN = r.ISBN
GROUP BY bcs.continent
ORDER BY rows_count DESC;

continent,rows_count,distinct_isbns,explicit_ratings
Other,1149780,340556,22429646


In [0]:
SELECT
    continent,
    COUNT(*) AS users
FROM user_geography
GROUP BY continent
ORDER BY users DESC;

continent,users
North America,161714
Europe,80058
Oceania,14795
null,9725
Asia,8385
South America,3507
Africa,675


In [0]:
CREATE OR REPLACE TABLE workspace.default.book_continent_stats AS
SELECT
    r.ISBN,
    g.continent,
    COUNT(*) AS explicit_ratings,
    COUNT(DISTINCT r.`User-ID`) AS unique_users,
    AVG(r.`Book-Rating`) AS avg_rating
FROM ratings r
INNER JOIN user_geography g
    ON r.`User-ID` = g.`User-ID`
WHERE r.`Book-Rating` > 0
  AND g.continent IS NOT NULL
GROUP BY
    r.ISBN,
    g.continent;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
    continent,
    COUNT(*) AS isbn_continent_rows,
    COUNT(DISTINCT ISBN) AS distinct_isbns,
    SUM(explicit_ratings) AS explicit_ratings
FROM workspace.default.book_continent_stats
GROUP BY continent
ORDER BY explicit_ratings DESC;

continent,isbn_continent_rows,distinct_isbns,explicit_ratings
North America,123338,123338,308887
Europe,56275,56275,80509
Oceania,10102,10102,12397
Asia,5320,5320,6134
South America,1129,1129,1197
Africa,196,196,200


In [0]:
WITH book_reach AS (
    SELECT
        s.ISBN,
        COUNT(DISTINCT s.continent) AS continents_reached
    FROM workspace.default.book_continent_stats s
    INNER JOIN eligible_books e
        ON s.ISBN = e.ISBN
    GROUP BY s.ISBN
)

SELECT
    continents_reached,
    COUNT(*) AS number_of_books,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_books
FROM book_reach
GROUP BY continents_reached
ORDER BY continents_reached;

continents_reached,number_of_books,percentage_of_books
1,7795,53.63
2,4533,31.19
3,1701,11.70
4,465,3.20
5,40,0.28
6,1,0.01


In [0]:
SELECT
    s.ISBN,
    COUNT(DISTINCT s.continent) AS continents_reached
FROM workspace.default.book_continent_stats s
INNER JOIN eligible_books e
    ON s.ISBN = e.ISBN
GROUP BY s.ISBN
ORDER BY continents_reached DESC
LIMIT 30;

ISBN,continents_reached
0971880107,6
0671004107,5
0062502182,5
0141000198,5
0446608955,5
0140481389,5
0440234743,5
0446603929,5
0811824829,5
0425109720,5


In [0]:
WITH book_reach AS (
    SELECT
        s.ISBN,
        COUNT(DISTINCT s.continent) AS continents_reached
    FROM workspace.default.book_continent_stats s
    INNER JOIN eligible_books e
        ON s.ISBN = e.ISBN
    GROUP BY s.ISBN
)

SELECT
    continents_reached,
    COUNT(*) AS number_of_books,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_books
FROM book_reach
GROUP BY continents_reached
ORDER BY continents_reached;

continents_reached,number_of_books,percentage_of_books
1,7795,53.63
2,4533,31.19
3,1701,11.70
4,465,3.20
5,40,0.28
6,1,0.01


Identifying the books with demonstrated international potential by measuring how many distinct languages their editions are available in.

In [0]:
CREATE OR REPLACE TABLE workspace.default.language_candidates AS
SELECT
    q.ISBN,
    b.`Book-Title` AS title,
    b.`Book-Author` AS author,
    q.rating_count,
    q.avg_rating,
    q.weighted_rating
FROM book_quality_scores q
INNER JOIN books b
    ON q.ISBN = b.ISBN
ORDER BY q.weighted_rating DESC
LIMIT 1000;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT ISBN) AS distinct_isbns,
    COUNT(DISTINCT title) AS distinct_titles,
    COUNT(DISTINCT author) AS distinct_authors
FROM workspace.default.language_candidates;

rows,distinct_isbns,distinct_titles,distinct_authors
1000,1000,933,609


In [0]:
SELECT *
FROM workspace.default.language_candidates
LIMIT 5;

ISBN,title,author,rating_count,avg_rating,weighted_rating
0439425220,Harry Potter and the Chamber of Secrets Postcard Book,J. K. Rowling,23,9.869565217391305,9.476857142857144
0618002235,"The Two Towers (The Lord of the Rings, Part 2)",J. R. R. Tolkien,25,9.72,9.378400000000001
0060256656,The Giving Tree,Shel Silverstein,20,9.75,9.33408
0836213319,Dilbert: A Book of Postcards,Scott Adams,13,9.923076923076923,9.297333333333334
0345339738,"The Return of the King (The Lord of the Rings, Part 3)",J.R.R. TOLKIEN,77,9.402597402597403,9.296975609756096


In [0]:
SELECT *
FROM workspace.default.language_candidates
ORDER BY weighted_rating DESC;

ISBN,title,author,rating_count,avg_rating,weighted_rating
0439425220,Harry Potter and the Chamber of Secrets Postcard Book,J. K. Rowling,23,9.869565217391305,9.476857142857144
0618002235,"The Two Towers (The Lord of the Rings, Part 2)",J. R. R. Tolkien,25,9.72,9.378400000000001
0060256656,The Giving Tree,Shel Silverstein,20,9.75,9.33408
0836213319,Dilbert: A Book of Postcards,Scott Adams,13,9.923076923076923,9.297333333333334
0345339738,"The Return of the King (The Lord of the Rings, Part 3)",J.R.R. TOLKIEN,77,9.402597402597403,9.296975609756096
1888054557,Postmarked Yesteryear: 30 Rare Holiday Postcards,Pamela E. Apkarian-Russell,11,10.0,9.272
0836220889,Calvin and Hobbes,Bill Watterson,24,9.583333333333334,9.25351724137931
0394800389,Fox in Socks (I Can Read It All by Myself Beginner Books),Dr. Seuss,14,9.785714285714286,9.229052631578947
0836218221,The Authoritative Calvin and Hobbes (Calvin and Hobbes),Bill Watterson,20,9.6,9.21408
0439139597,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,137,9.262773722627736,9.206704225352112


In [0]:
%python
import requests
import pandas as pd
import time

# Get all 1,000 candidate books
candidates = spark.sql("""
    SELECT ISBN, title, author
    FROM workspace.default.language_candidates
""").toPandas()

results = []

headers = {
    "User-Agent": "BookshelfInspectorStudentProject/1.0"
}

for i, row in candidates.iterrows():

    isbn = str(row["ISBN"]).strip()

    params = {
        "q": f"isbn:{isbn}",
        "fields": "key,title,author_name,edition_count,language",
        "limit": 1
    }

    try:
        response = requests.get(
            "https://openlibrary.org/search.json",
            params=params,
            headers=headers,
            timeout=20
        )

        response.raise_for_status()
        data = response.json()

        if data.get("docs"):

            book = data["docs"][0]

            languages = book.get("language", [])
            languages = sorted(set(languages))

            results.append({
                "ISBN": isbn,
                "work_id": book.get("key"),
                "number_of_editions": book.get("edition_count"),
                "number_of_languages": len(languages),
                "languages": ", ".join(languages)
            })

        else:

            results.append({
                "ISBN": isbn,
                "work_id": None,
                "number_of_editions": None,
                "number_of_languages": None,
                "languages": None
            })

    except Exception as e:

        print(f"Error for ISBN {isbn}: {e}")

        results.append({
            "ISBN": isbn,
            "work_id": None,
            "number_of_editions": None,
            "number_of_languages": None,
            "languages": None
        })

    # Show progress
    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1} / {len(candidates)} books")

    time.sleep(1)

# Convert result to pandas dataframe
result_df = pd.DataFrame(results)

# Convert to Spark dataframe
spark_result = spark.createDataFrame(result_df)

# Save permanently in Databricks
spark_result.write.mode("overwrite").saveAsTable(
    "workspace.default.openlibrary_languages"
)

print("Finished!")
print(f"Processed: {len(result_df)} books")

display(result_df.head(20))

Processed 50 / 1000 books
Processed 100 / 1000 books
Processed 150 / 1000 books
Processed 200 / 1000 books
Processed 250 / 1000 books
Processed 300 / 1000 books
Processed 350 / 1000 books
Processed 400 / 1000 books
Processed 450 / 1000 books
Processed 500 / 1000 books
Processed 550 / 1000 books
Processed 600 / 1000 books
Processed 650 / 1000 books
Processed 700 / 1000 books
Processed 750 / 1000 books
Processed 800 / 1000 books
Processed 850 / 1000 books
Processed 900 / 1000 books
Processed 950 / 1000 books
Processed 1000 / 1000 books
Finished!
Processed: 1000 books


ISBN,work_id,number_of_editions,number_of_languages,languages
0439425220,/works/OL8120106W,1.0,1.0,eng
0618002235,/works/OL27479W,290.0,27.0,"afr, ast, baq, bul, cat, chi, cze, dan, dut, eng, fre, ger, gre, heb, hrv, hun, ind, ita, jpn, pol, por, rus, slv, spa, swe, tur, yid"
0060256656,/works/OL3368231W,61.0,12.0,"ara, chi, eng, fre, heb, ita, jpn, kor, lat, por, spa, vie"
0836213319,/works/OL273369W,27.0,2.0,"eng, spa"
0345339738,/works/OL27448W,253.0,18.0,"bul, cat, cze, dut, eng, fre, ger, glg, ita, jpn, kor, pol, por, rus, slo, spa, swe, tur"
1888054557,/works/OL8473465W,2.0,1.0,eng
0836220889,/works/OL16050250W,10.0,2.0,"eng, fre"
0394800389,/works/OL261159W,31.0,3.0,"chi, eng, spa"
0836218221,/works/OL3004222W,14.0,2.0,"eng, spa"
0439139597,/works/OL82560W,244.0,33.0,"afr, alb, ara, arm, ben, bul, chi, cze, dan, dut, eng, fre, ger, gre, heb, hin, hun, ita, jpn, kor, lit, mar, nor, pol, por, rum, rus, spa, swe, tha, tur, urd, vie"


In [0]:
SELECT
    COUNT(*) AS total_candidates,
    COUNT(work_id) AS matched_openlibrary,
    COUNT(*) - COUNT(work_id) AS not_found,
    COUNT(number_of_languages) AS with_language_data,
    ROUND(AVG(number_of_languages), 2) AS avg_languages,
    MAX(number_of_languages) AS max_languages
FROM workspace.default.openlibrary_languages;

total_candidates,matched_openlibrary,not_found,with_language_data,avg_languages,max_languages
1000,996,4,996,7.0,51.0


In [0]:
SELECT
    c.title,
    c.author,
    l.ISBN,
    l.work_id,
    l.number_of_editions,
    l.number_of_languages,
    l.languages
FROM workspace.default.openlibrary_languages l
INNER JOIN workspace.default.language_candidates c
    ON l.ISBN = c.ISBN
ORDER BY l.number_of_languages DESC
LIMIT 30;

title,author,ISBN,work_id,number_of_editions,number_of_languages,languages
Alice's Adventures in Wonderland/With All the Original Illustrations by Sir John Tenniel (Dover Thrift Editions),Lewis Carroll,0486275434,/works/OL138052W,3547.0,51.0,"alt, ara, bel, bre, bul, cat, ceb, chi, cor, crp, cze, dan, dut, eng, epo, fin, fre, fry, ger, gle, gre, guj, haw, heb, hrv, hun, ita, jpn, kaz, kir, kor, lad, lat, lav, orm, pol, por, rus, sco, slo, spa, srp, swa, swe, syr, tel, tur, ukr, und, wel, yid"
Hamlet,William Shakespeare,067172262X,/works/OL9170454W,2377.0,50.0,"alb, amh, ara, arm, baq, bul, cat, chi, cze, dan, dut, eng, epo, fao, fin, fre, fry, ger, gre, guj, heb, hun, ice, ina, ita, jpn, kur, lit, mlt, mul, per, pol, por, rum, rus, san, sin, slo, spa, srp, swe, tgl, tib, tlh, tur, ukr, und, urd, wel, yid"
Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,0590353403,/works/OL82563W,400.0,49.0,"afr, alb, ara, ben, bul, cat, chi, cze, dan, dut, eng, fin, fre, ger, gla, gle, grc, gre, heb, hin, hrv, hun, ice, ita, jpn, kal, kor, lat, lav, lit, ltz, mar, nep, nor, per, pol, por, rum, rus, spa, swe, tha, tib, tur, ukr, urd, vie, wel, yid"
Harry Potter and the Sorcerer's Stone (Book 1 Audio CD),J. K. Rowling,0807281956,/works/OL82563W,400.0,49.0,"afr, alb, ara, ben, bul, cat, chi, cze, dan, dut, eng, fin, fre, ger, gla, gle, grc, gre, heb, hin, hrv, hun, ice, ita, jpn, kal, kor, lat, lav, lit, ltz, mar, nep, nor, per, pol, por, rum, rus, spa, swe, tha, tib, tur, ukr, urd, vie, wel, yid"
Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,043936213X,/works/OL82563W,400.0,49.0,"afr, alb, ara, ben, bul, cat, chi, cze, dan, dut, eng, fin, fre, ger, gla, gle, grc, gre, heb, hin, hrv, hun, ice, ita, jpn, kal, kor, lat, lav, lit, ltz, mar, nep, nor, per, pol, por, rum, rus, spa, swe, tha, tib, tur, ukr, urd, vie, wel, yid"
Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback)),J. K. Rowling,059035342X,/works/OL82563W,400.0,49.0,"afr, alb, ara, ben, bul, cat, chi, cze, dan, dut, eng, fin, fre, ger, gla, gle, grc, gre, heb, hin, hrv, hun, ice, ita, jpn, kal, kor, lat, lav, lit, ltz, mar, nep, nor, per, pol, por, rum, rus, spa, swe, tha, tib, tur, ukr, urd, vie, wel, yid"
The Little Prince (Wordsworth Collection),Antoine de Saint-ExupÃ©ry,1853261580,/works/OL10263W,699.0,40.0,"amh, arm, bre, cat, chi, cze, dan, eng, epo, fao, fre, ger, gle, grc, gsw, heb, hrv, ina, ita, jpn, kok, kor, lat, mwl, ota, per, pol, por, roh, rus, spa, swe, tgl, tur, und, vie, wel, yid, yue, zul"
The Little Prince,Antoine de Saint-ExupÃ©ry,0156528207,/works/OL10263W,699.0,40.0,"amh, arm, bre, cat, chi, cze, dan, eng, epo, fao, fre, ger, gle, grc, gsw, heb, hrv, ina, ita, jpn, kok, kor, lat, mwl, ota, per, pol, por, roh, rus, spa, swe, tgl, tur, und, vie, wel, yid, yue, zul"
The Little Prince,Antoine de Saint-ExupÃ©ry,0152023984,/works/OL10263W,699.0,40.0,"amh, arm, bre, cat, chi, cze, dan, eng, epo, fao, fre, ger, gle, grc, gsw, heb, hrv, ina, ita, jpn, kok, kor, lat, mwl, ota, per, pol, por, roh, rus, spa, swe, tgl, tur, und, vie, wel, yid, yue, zul"
Der Kleine Prinz Prince German Hardy Boys,Antoine de Saint-ExupÃ©ry,3792000261,/works/OL10263W,699.0,40.0,"amh, arm, bre, cat, chi, cze, dan, eng, epo, fao, fre, ger, gle, grc, gsw, heb, hrv, ina, ita, jpn, kok, kor, lat, mwl, ota, per, pol, por, roh, rus, spa, swe, tgl, tur, und, vie, wel, yid, yue, zul"


In [0]:
CREATE OR REPLACE TEMP VIEW international_book_works AS

SELECT
    l.work_id,

    FIRST(c.title) AS title,
    FIRST(c.author) AS author,

    COUNT(DISTINCT c.ISBN) AS candidate_isbns,

    MAX(c.weighted_rating) AS best_weighted_rating,
    AVG(c.weighted_rating) AS avg_weighted_rating,
    SUM(c.rating_count) AS total_explicit_ratings,

    MAX(l.number_of_editions) AS openlibrary_editions,
    MAX(l.number_of_languages) AS number_of_languages,
    FIRST(l.languages) AS languages

FROM workspace.default.language_candidates c

INNER JOIN workspace.default.openlibrary_languages l
    ON c.ISBN = l.ISBN

WHERE l.work_id IS NOT NULL

GROUP BY l.work_id;

In [0]:
SELECT
    title,
    author,
    work_id,
    candidate_isbns,
    total_explicit_ratings,
    ROUND(avg_weighted_rating, 2) AS avg_quality,
    number_of_languages,
    openlibrary_editions,
    languages
FROM international_book_works
ORDER BY number_of_languages DESC, avg_weighted_rating DESC
LIMIT 30;

title,author,work_id,candidate_isbns,total_explicit_ratings,avg_quality,number_of_languages,openlibrary_editions,languages
Alice's Adventures in Wonderland/With All the Original Illustrations by Sir John Tenniel (Dover Thrift Editions),Lewis Carroll,/works/OL138052W,1,6,8.4,51.0,3547.0,"alt, ara, bel, bre, bul, cat, ceb, chi, cor, crp, cze, dan, dut, eng, epo, fin, fre, fry, ger, gle, gre, guj, haw, heb, hrv, hun, ita, jpn, kaz, kir, kor, lad, lat, lav, orm, pol, por, rus, sco, slo, spa, srp, swa, swe, syr, tel, tur, ukr, und, wel, yid"
Hamlet,William Shakespeare,/works/OL9170454W,1,24,8.53,50.0,2377.0,"alb, amh, ara, arm, baq, bul, cat, chi, cze, dan, dut, eng, epo, fao, fin, fre, fry, ger, gre, guj, heb, hun, ice, ina, ita, jpn, kur, lit, mlt, mul, per, pol, por, rum, rus, san, sin, slo, spa, srp, swe, tgl, tib, tlh, tur, ukr, und, urd, wel, yid"
Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback)),J. K. Rowling,/works/OL82563W,4,490,8.86,49.0,400.0,"afr, alb, ara, ben, bul, cat, chi, cze, dan, dut, eng, fin, fre, ger, gla, gle, grc, gre, heb, hin, hrv, hun, ice, ita, jpn, kal, kor, lat, lav, lit, ltz, mar, nep, nor, per, pol, por, rum, rus, spa, swe, tha, tib, tur, ukr, urd, vie, wel, yid"
Der Kleine Prinz Prince German Hardy Boys,Antoine de Saint-ExupÃ©ry,/works/OL10263W,9,156,8.69,40.0,699.0,"amh, arm, bre, cat, chi, cze, dan, eng, epo, fao, fre, ger, gle, grc, gsw, heb, hrv, ina, ita, jpn, kok, kor, lat, mwl, ota, per, pol, por, roh, rus, spa, swe, tgl, tur, und, vie, wel, yid, yue, zul"
Romeo and Juliet (Bantam Classic),William Shakespeare,/works/OL362427W,1,10,8.42,38.0,985.0,"afr, ara, arm, ben, cat, chi, dan, dut, eng, fin, fre, ger, gre, guj, heb, hin, hrv, hun, ice, ita, jpn, kor, mar, mul, nor, pol, por, rus, sin, spa, srp, tib, tuk, tur, ukr, urd, vie, wel"
Harry Potter y el prisionero de Azkaban,J. K. Rowling,/works/OL82536W,4,291,8.92,37.0,279.0,"alb, ara, arm, ben, bul, cat, chi, cze, dut, eng, fre, geo, ger, gre, heb, hin, hun, ita, jpn, kor, lit, mar, nob, per, pol, por, rum, rus, som, spa, srp, swe, tha, tib, tur, urd, vie"
King Lear (3rd Series),William Shakespeare,/works/OL259026W,1,6,8.76,37.0,1614.0,"afr, ara, arm, bel, bul, cat, chi, dut, eng, epo, fin, fre, geo, ger, gre, heb, hin, hrv, hun, ita, jpn, mul, ota, pan, pol, por, rus, slo, slv, spa, swe, tel, tur, ukr, und, urd, yid"
Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,/works/OL82537W,5,343,8.71,37.0,306.0,"alb, ara, arm, bul, cat, chi, cze, dan, dut, eng, fin, fre, geo, ger, gre, heb, hin, hun, ita, jpn, kor, lat, lit, mar, per, pol, por, rum, rus, spa, swe, tha, tib, tur, ukr, urd, vie"
Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,/works/OL82560W,3,257,9.02,33.0,244.0,"afr, alb, ara, arm, ben, bul, chi, cze, dan, dut, eng, fre, ger, gre, heb, hin, hun, ita, jpn, kor, lit, mar, nor, pol, por, rum, rus, spa, swe, tha, tur, urd, vie"
Harry Potter and the Order of the Phoenix (Book 5),J. K. Rowling,/works/OL82548W,1,206,9.0,32.0,247.0,"ara, arm, ben, bul, cat, chi, cze, dan, dut, eng, fre, geo, ger, heb, hin, hrv, hun, ita, jpn, kor, lit, per, pol, por, rum, rus, spa, swe, tha, tur, ukr, vie"


In [0]:
SELECT
    title,
    number_of_languages
FROM international_book_works
WHERE number_of_languages IS NOT NULL
ORDER BY number_of_languages DESC
LIMIT 15;

title,number_of_languages
Alice's Adventures in Wonderland/With All the Original Illustrations by Sir John Tenniel (Dover Thrift Editions),51.0
Hamlet,50.0
Harry Potter and the Sorcerer's Stone (Book 1),49.0
The Little Prince,40.0
Romeo and Juliet (Bantam Classic),38.0
Harry Potter and the Prisoner of Azkaban (Book 3),37.0
King Lear (3rd Series),37.0
Harry Potter and the Chamber of Secrets (Book 2),37.0
Harry Potter and the Goblet of Fire (Book 4),33.0
The Hobbit,32.0


In [0]:
CREATE OR REPLACE TABLE workspace.default.international_book_works AS

SELECT
    l.work_id,

    FIRST(c.title) AS title,
    FIRST(c.author) AS author,

    COUNT(DISTINCT c.ISBN) AS candidate_isbns,

    MAX(c.weighted_rating) AS best_weighted_rating,
    AVG(c.weighted_rating) AS avg_weighted_rating,
    SUM(c.rating_count) AS total_explicit_ratings,

    MAX(l.number_of_editions) AS openlibrary_editions,
    MAX(l.number_of_languages) AS number_of_languages,
    FIRST(l.languages) AS languages

FROM workspace.default.language_candidates c

INNER JOIN workspace.default.openlibrary_languages l
    ON c.ISBN = l.ISBN

WHERE l.work_id IS NOT NULL

GROUP BY l.work_id;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
    COUNT(*) AS number_of_works,
    MAX(number_of_languages) AS max_languages
FROM workspace.default.international_book_works;

number_of_works,max_languages
806,51.0
